# 2. Entrenamiento del modelo — Deteccion de fraude

Problema: **clasificacion binaria supervisada** (`isFraud`), fuertemente
desbalanceado (~0.1-0.3% de positivos).

Flujo:
1. Train/test split estratificado sobre la **referencia** (`step <= 400`).
2. **Balanceo de clases** con las tecnicas vistas en la Sesion 8 del curso
   (Reweighing y Oversampling), generalizadas de "equilibrar un grupo
   sensible" a "equilibrar la clase minoritaria del target".
3. Modelo base (Logistic Regression) como piso de comparacion.
4. Optimizacion de hiperparametros de un `XGBoostClassifier` con
   **Random Search** (`RandomizedSearchCV`, Sesion 3).
5. Registro de experimentos (parametros, metricas, artefactos) en **MLflow**.
6. Evaluacion **offline** (holdout de referencia) y **online** (datos de
   produccion `step > 400`, que en este dataset simulado si tienen
   `isFraud` real, permitiendo medir el desempeno "como si" ya estuviera
   desplegado).

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import joblib
import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, train_test_split
from xgboost import XGBClassifier

from src.balancing import oversample_minority, reweighting_sample_weights
from src.config import (
    MODEL_DIR,
    MODEL_PATH,
    PRODUCTION_DATA_PATH,
    RANDOM_STATE,
    REFERENCE_DATA_PATH,
    TARGET_COL,
)
from src.features import select_model_features
from src.metrics import classification_metrics
from src.tracking import init_mlflow, log_classification_metrics

MODEL_DIR.mkdir(parents=True, exist_ok=True)
init_mlflow()


In [ ]:
reference = pd.read_csv(REFERENCE_DATA_PATH)
production = pd.read_csv(PRODUCTION_DATA_PATH)

FEATURES = select_model_features(reference)
print(f"{len(FEATURES)} features:", FEATURES)

X = reference[FEATURES]
y = reference[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print(X_train.shape, X_test.shape, y_train.mean(), y_test.mean())

## 2.1 Balanceo de clases (tecnicas de la Sesion 8)

El curso introduce dos tecnicas de *pre-processing* para dar mas peso o
representacion a la clase minoritaria, en el contexto de **fairness**
(equilibrar un grupo demografico sensible):

- **Reweighing** (`mod0_Reweighing.ipynb`): pesos pasados como
  `sample_weight` a `.fit()` (en el notebook original, 2.0 para el grupo
  desfavorecido y 1.0 para el resto).
- **Oversampling** (`mod1_Oversampling.ipynb`): duplicar con reemplazo las
  filas del grupo minoritario via `sklearn.utils.resample`.

Aqui generalizamos exactamente la misma mecanica, pero equilibrando la
clase minoritaria del **target** (`isFraud`) en vez de un grupo sensible
(ver [`src/balancing.py`](../src/balancing.py)). No se usa ninguna libreria
de balanceo externa (p.ej. `imbalanced-learn`/SMOTE) por no haber sido
parte del curso.

Comparamos, sobre un mismo holdout y un mismo algoritmo
(`RandomForestClassifier`, igual que en los notebooks de la Sesion 8),
tres escenarios: **Original (sin balanceo)**, **Reweighing** y
**Oversampling**, usando PR-AUC y Recall (las metricas que mas importan
cuando la clase positiva es rara).

In [ ]:
def eval_estrategia(nombre, model, X_tr, y_tr, sample_weight=None):
    model.fit(X_tr, y_tr, sample_weight=sample_weight)
    proba = model.predict_proba(X_test)[:, 1]
    preds = (proba >= 0.5).astype(int)
    m = classification_metrics(y_test, preds, proba)
    m["estrategia"] = nombre
    return m


rf_kwargs = dict(n_estimators=200, max_depth=8, random_state=RANDOM_STATE, n_jobs=-1)

# --- Original (sin balanceo) ---
resultados = [eval_estrategia("Original", RandomForestClassifier(**rf_kwargs), X_train, y_train)]

# --- Reweighing ---
weights = reweighting_sample_weights(y_train)
resultados.append(
    eval_estrategia("Reweighing", RandomForestClassifier(**rf_kwargs), X_train, y_train, sample_weight=weights)
)

# --- Oversampling ---
X_train_over, y_train_over = oversample_minority(X_train, y_train, random_state=RANDOM_STATE)
resultados.append(eval_estrategia("Oversampling", RandomForestClassifier(**rf_kwargs), X_train_over, y_train_over))

comparacion = pd.DataFrame(resultados).set_index("estrategia")[
    ["precision", "recall", "f1", "pr_auc", "roc_auc"]
]
comparacion

**Decision:** seguimos adelante con la estrategia con mejor `pr_auc` en la
comparacion anterior. Si gana **Reweighing**, la llevamos tal cual a la
optimizacion de hiperparametros (los pesos se pasan por fold dentro de la
validacion cruzada, sin riesgo de fuga de informacion). Si gana
**Oversampling**, usamos el set de entrenamiento ya sobremuestreado para
el split (nota: por simplicidad no se re-aplica oversampling dentro de
cada fold de la validacion cruzada; es una limitacion conocida de esta
tecnica que se documenta en las conclusiones).

In [ ]:
mejor_estrategia = comparacion["pr_auc"].idxmax()
print("Estrategia de balanceo elegida:", mejor_estrategia)

if mejor_estrategia == "Reweighing":
    X_train_bal, y_train_bal = X_train, y_train
    sample_weight_bal = reweighting_sample_weights(y_train)
elif mejor_estrategia == "Oversampling":
    X_train_bal, y_train_bal = X_train_over, y_train_over
    sample_weight_bal = None
else:
    X_train_bal, y_train_bal = X_train, y_train
    sample_weight_bal = None

with mlflow.start_run(run_name="class_balancing_comparison"):
    mlflow.log_param("balancing_strategy_chosen", mejor_estrategia)
    for estrategia, row in comparacion.iterrows():
        for metric_name, value in row.items():
            mlflow.log_metric(f"{estrategia.lower()}_{metric_name}", float(value))

## 2.2 Modelo base (Logistic Regression)

Sirve como piso de comparacion, ya usando la estrategia de balanceo elegida arriba.

In [ ]:
with mlflow.start_run(run_name="baseline_logreg"):
    baseline = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
    baseline.fit(X_train_bal, y_train_bal, sample_weight=sample_weight_bal)

    proba = baseline.predict_proba(X_test)[:, 1]
    preds = (proba >= 0.5).astype(int)
    metrics = classification_metrics(y_test, preds, proba)

    mlflow.log_param("model", "LogisticRegression")
    mlflow.log_param("balancing_strategy", mejor_estrategia)
    log_classification_metrics("offline_", metrics)
    print(metrics)

## 2.3 Optimizacion de hiperparametros - XGBoost

Tecnica vista en el curso (Sesion 3): **Random Search** (`RandomizedSearchCV`),
exploracion aleatoria del espacio de hiperparametros (en vez de probar la
grilla completa como Grid Search).

El desbalance de clases ya se maneja con la estrategia elegida en 2.1
(Reweighing u Oversampling) - no se usa `scale_pos_weight` ni
`class_weight` (no vistos en el curso).


In [ ]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

param_dist = {
    "n_estimators": [200, 300, 400, 600],
    "max_depth": [3, 4, 5, 6, 8],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "min_child_weight": [1, 3, 5],
}

base_xgb = XGBClassifier(
    objective="binary:logistic",
    eval_metric="aucpr",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

fit_params = {"sample_weight": sample_weight_bal} if sample_weight_bal is not None else {}

random_search = RandomizedSearchCV(
    base_xgb,
    param_distributions=param_dist,
    n_iter=15,
    scoring="average_precision",
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
)
random_search.fit(X_train_bal, y_train_bal, **fit_params)
print("Mejor PR-AUC (CV) Random Search:", random_search.best_score_)
print(random_search.best_params_)

## 2.4 Modelo campeon ('champion')

Se usan los mejores hiperparametros encontrados por Random Search, y se entrena sobre todo el train set balanceado.

In [ ]:
best_params = random_search.best_params_
print("Mejores hiperparametros (Random Search):", best_params)

champion = XGBClassifier(
    objective="binary:logistic",
    eval_metric="aucpr",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    **best_params,
)
champion.fit(X_train_bal, y_train_bal, sample_weight=sample_weight_bal)


In [ ]:
with mlflow.start_run(run_name="champion_xgboost_random_search"):
    mlflow.log_param("model", "XGBClassifier")
    mlflow.log_param("hpo_method", "random_search")
    mlflow.log_param("balancing_strategy", mejor_estrategia)
    for k, v in best_params.items():
        mlflow.log_param(k, v)

    # --- Evaluacion OFFLINE (holdout de referencia, step <= 400) ---
    proba_test = champion.predict_proba(X_test)[:, 1]
    preds_test = (proba_test >= 0.5).astype(int)
    offline_metrics = classification_metrics(y_test, preds_test, proba_test)
    log_classification_metrics("offline_", offline_metrics)
    print("OFFLINE:", offline_metrics)

    # --- Evaluacion ONLINE (produccion, step > 400) ---
    X_prod, y_prod = production[FEATURES], production[TARGET_COL]
    proba_prod = champion.predict_proba(X_prod)[:, 1]
    preds_prod = (proba_prod >= 0.5).astype(int)
    online_metrics = classification_metrics(y_prod, preds_prod, proba_prod)
    log_classification_metrics("online_", online_metrics)
    print("ONLINE:", online_metrics)

    mlflow.xgboost.log_model(champion, artifact_path="model")

    cm = confusion_matrix(y_test, preds_test)
    fig, ax = plt.subplots(figsize=(4, 4))
    ConfusionMatrixDisplay(cm, display_labels=["No fraude", "Fraude"]).plot(ax=ax, cmap="Blues")
    fig.savefig("../reports/confusion_matrix_offline.png", bbox_inches="tight")
    mlflow.log_artifact("../reports/confusion_matrix_offline.png")
    plt.show()

In [ ]:
joblib.dump({"model": champion, "features": FEATURES}, MODEL_PATH)
print("Modelo guardado en", MODEL_PATH)

production = production.copy()
production["fraud_proba"] = champion.predict_proba(production[FEATURES])[:, 1]
production["fraud_pred"] = (production["fraud_proba"] >= 0.5).astype(int)
production.to_csv(PRODUCTION_DATA_PATH.parent / "production_scored.csv", index=False)
print("Produccion scoreada guardada para el notebook de monitoreo.")

## 2.5 Siguiente paso

- Ver experimentos: `mlflow ui --backend-store-uri ./mlruns` (desde la raiz del repo).
- Continuar con [`3.0-interpretabilidad.ipynb`](3.0-interpretabilidad.ipynb).